# Exercise 10 - Dropout and Batch Normalization

Estimated time: **35-40 minutes**

The goal of this exercise is to experiment with dropout and batch normalization using a simple CNN and the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset, which is described in detail in the paper [Learning Multiple Layers of Features from Tiny Images](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf), Alex Krizhevsky, 2009. The [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset consists of 60,000 32x32 pixel colour images in 10 classes, with 6,000 images per class. There are 50,000 training images and 10,000 test images.

- Use **T4 GPU** as hardware accelerator for this exercise

First run the code below to build and train the CNN without dropout or batch normalization to establish a benchmark.



In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import tensorflow
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Activation, MaxPooling2D, Dense, Flatten, Input
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.utils  import to_categorical

# Ensure a clean start
tensorflow.keras.backend.clear_session()

# The data, shuffled and split between train and test sets:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Plot some images as a sanity check
fig = plt.figure(1, figsize=(15,1))
m = 10
for i in range(m):
    a = fig.add_subplot(1,m,i+1)
    plt.imshow(x_train[i])
plt.show()

n_labels = 10

# Convert class vectors to binary class matrices.
y_train = to_categorical(y_train, n_labels)
y_test = to_categorical(y_test, n_labels)

# Normalize the images to have zero mean and values in range [-1,+1]
x_train = x_train.astype('float32') / 255
x_test  = x_test.astype('float32') / 255
mean    = x_train.mean()
x_train = x_train - mean
x_test  = x_test - mean

model = Sequential()
model.add(Input(shape=x_train.shape[1:]))
model.add(Conv2D(32, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(Conv2D(32, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dense(n_labels))
model.add(Activation('softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=32, epochs=10,
          validation_data=(x_test, y_test), shuffle=True);

Now add dropout and batch normalization to the model. You should experiment with adding dropout and batch normalization separately so you can see their individual effects as well as trying them both together. You would typically apply dropout at the outputs of layers that have a large number of units and dense connections (which implies a high degree of variance). You can select different degrees of dropout. For example, `Dropout(0.25)` drops a randomly selected 25% of the units in a layer, then multiplies the activations of that layer by 4/3 in the final trained model. You would typically apply batch normalization at the inputs of non-linear activation functions in order to keep the inputs to the non-linearity around the sweet spot.

Which of the two, dropout or batch normalization, has the stronger regularization effect?

In [ ]:
#

#### Solution

Here is our answer. Do not run the cell below unless you want to see the answer we provide!

<details>
    <summary> Click here for the answer</summary>

    import numpy as np
    import matplotlib.pyplot as plt
    import tensorflow
    from tensorflow.keras.datasets import cifar10
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv2D, Activation, MaxPooling2D, Dense, Flatten
    from tensorflow.keras.layers import Dropout, BatchNormalization
    from tensorflow.keras.utils  import to_categorical
    from tensorflow.keras.backend  import clear_session

    def build_and_run_model(dropout = False, bn = False):

        # Ensure a clean start
        clear_session()

        # The data, shuffled and split between train and test sets:
        (x_train, y_train), (x_test, y_test) = cifar10.load_data()

        n_labels = 10

        # Convert class vectors to binary class matrices.
        y_train = to_categorical(y_train, n_labels)
        y_test = to_categorical(y_test, n_labels)

        # Normalize the images to have zero mean and values in range [-1,+1]
        x_train = x_train.astype('float32') / 255
        x_test  = x_test.astype('float32') / 255
        mean    = x_train.mean()
        x_train = x_train - mean
        x_test  = x_test - mean

        model = Sequential()
        model.add(Input(shape=x_train.shape[1:]))
        model.add(Conv2D(32, (3, 3), padding='same',  use_bias=(not bn)))
        if bn: model.add(BatchNormalization(scale=False))
        model.add(Activation('relu'))
        model.add(Conv2D(32, (3, 3), padding='same', use_bias=(not bn)))
        if bn: model.add(BatchNormalization(scale=False))
        model.add(Activation('relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))

        model.add(Conv2D(64, (3, 3), padding='same', use_bias=(not bn)))
        if bn: model.add(BatchNormalization(scale=False))
        model.add(Activation('relu'))
        model.add(Conv2D(64, (3, 3), use_bias=(not bn)))
        if bn: model.add(BatchNormalization(scale=False))
        model.add(Activation('relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        if dropout: model.add(Dropout(0.5))

        model.add(Flatten())
        model.add(Dense(512, use_bias=(not bn)))
        if bn: model.add(BatchNormalization(scale=False))
        model.add(Activation('relu'))
        if dropout: model.add(Dropout(0.5))

        model.add(Dense(n_labels))
        model.add(Activation('softmax'))
        model.summary()

        model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

        model.fit(x_train, y_train,
              batch_size=32, epochs=10,
              validation_data=(x_test, y_test), shuffle=True);

    print('\nDropout only')
    build_and_run_model(dropout = True,  bn = False)

    print('\nBatch normalization only')
    build_and_run_model(dropout = False, bn = True)

    print('\nBoth dropout and batch normalization')
    build_and_run_model(dropout = True,  bn = True)
    
</details>